In [42]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
import m2cgen as m2c
import numpy as np
import collections

In [38]:
df=pd.read_csv('C:/Users/bogda/gazpromhack/data/1.csv', encoding='cp1251', sep=';')
df.head()

,TimeStamp,Температура масла в магистрали общей откачки (поз. Тм),Температура слива масла из опоры турбины (поз. Т606),Температура масла на входе в двигатель за фильтром (поз. Т607),Температура топливного газа перед СК (перед расходомерным устройством) (поз. Ттг),Давление масла в нагнетающей магистрали двигателя (после фильтра) (поз. Pм),Давление масла в магистрали общей откачки (поз. Р615),Давление за ОК Pk1,Давление за ОК Pk2,Давление топливного газа перед дозаторами (поз. Pтг1.1),...,Частота вращения РНД (Расчетная),Частота вращения РВД (Расчетная),Частота вращения РСТ (Расчетная),Температура воздуха на входе в газогенератор (Расчетная),Давление воздуха за компрессором (Расчетная),Частота вращения РНД приведенная,Частота вращения РВД приведенная,Температура газов за ТНД максимальная (Расчетная),Положение лопаток НА КВД. (поз. A2),Положение КПВ (поз. A3)
0,12:10:00.001,21.548,28.16,24.361,8.762,302.095,198.239,244.713,257.590,2917.776,...,4052.990,9564.946,1490.427,5.601,250.959,4119.418,9722.291,439.119,16.139,8.072
1,12:10:00.024,21.548,28.16,24.361,8.762,302.095,198.239,244.713,257.590,2917.776,...,4053.209,9565.001,1490.403,5.601,251.152,4119.683,9722.339,439.119,16.139,8.072
2,12:10:00.046,21.548,28.16,24.361,8.762,302.559,198.220,243.865,258.663,2916.046,...,4053.209,9565.001,1490.403,5.601,251.152,4119.905,9722.395,439.119,15.876,7.931
3,12:10:00.068,21.548,28.16,24.361,8.762,302.559,198.220,243.865,258.663,2916.046,...,4053.079,9564.868,1490.337,5.601,251.264,4119.905,9722.395,439.107,15.876,7.931
4,12:10:00.091,21.548,28.23,24.361,8.762,302.262,197.553,243.865,257.820,2915.057,...,4053.079,9564.868,1490.337,5.601,251.264,4119.773,9722.260,439.107,15.758,7.931


In [39]:
df=df.drop('TimeStamp',axis=1)
df

,Температура масла в магистрали общей откачки (поз. Тм),Температура слива масла из опоры турбины (поз. Т606),Температура масла на входе в двигатель за фильтром (поз. Т607),Температура топливного газа перед СК (перед расходомерным устройством) (поз. Ттг),Давление масла в нагнетающей магистрали двигателя (после фильтра) (поз. Pм),Давление масла в магистрали общей откачки (поз. Р615),Давление за ОК Pk1,Давление за ОК Pk2,Давление топливного газа перед дозаторами (поз. Pтг1.1),Вибрация промежуточного корпуса газогенератора (гориз.) (поз. В1),...,Частота вращения РНД (Расчетная),Частота вращения РВД (Расчетная),Частота вращения РСТ (Расчетная),Температура воздуха на входе в газогенератор (Расчетная),Давление воздуха за компрессором (Расчетная),Частота вращения РНД приведенная,Частота вращения РВД приведенная,Температура газов за ТНД максимальная (Расчетная),Положение лопаток НА КВД. (поз. A2),Положение КПВ (поз. A3)
0,21.548,28.160,24.361,8.762,302.095,198.239,244.713,257.590,2917.776,1.977,...,4052.990,9564.946,1490.427,5.601,250.959,4119.418,9722.291,439.119,16.139,8.072
1,21.548,28.160,24.361,8.762,302.095,198.239,244.713,257.590,2917.776,1.977,...,4053.209,9565.001,1490.403,5.601,251.152,4119.683,9722.339,439.119,16.139,8.072
2,21.548,28.160,24.361,8.762,302.559,198.220,243.865,258.663,2916.046,1.972,...,4053.209,9565.001,1490.403,5.601,251.152,4119.905,9722.395,439.119,15.876,7.931
3,21.548,28.160,24.361,8.762,302.559,198.220,243.865,258.663,2916.046,1.972,...,4053.079,9564.868,1490.337,5.601,251.264,4119.905,9722.395,439.107,15.876,7.931
4,21.548,28.230,24.361,8.762,302.262,197.553,243.865,257.820,2915.057,1.977,...,4053.079,9564.868,1490.337,5.601,251.264,4119.773,9722.260,439.107,15.758,7.931
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12654,43.561,65.943,23.177,23.477,308.473,199.926,935.612,941.988,2859.809,3.783,...,7699.703,10666.571,3697.968,4.822,939.183,7834.907,10856.534,586.625,79.420,9.020
12655,43.561,65.943,23.107,23.477,308.473,199.926,933.762,942.065,2860.674,3.786,...,7700.150,10667.598,3698.198,4.822,938.800,7837.355,10857.264,586.625,79.667,9.014
12656,43.561,65.943,23.107,23.477,308.473,199.926,933.762,942.065,2860.674,3.786,...,7701.266,10666.765,3698.287,4.822,937.913,7837.810,10858.309,586.625,79.667,9.014
12657,100.000,65.943,23.107,23.477,308.232,200.667,934.841,942.065,2857.584,3.778,...,7701.266,10666.765,3698.287,4.822,937.913,7838.946,10857.461,586.625,79.637,8.975


In [ ]:
class TrendingCusumDetector:
    def __init__(self, target_slope, sigma_slope, h_factor=5):
        self.target_slope = target_slope
        self.K = 0.5 * sigma_slope
        self.H = h_factor * sigma_slope
        
        self.s_high = 0.0
        self.s_low = 0.0
        self.last_val = None

    def update(self, val):
        if self.last_val is None:
            self.last_val = val
            return False, 0, 0
        
        current_slope = val - self.last_val
        self.last_val = val
        
        diff = current_slope - self.target_slope
        
        self.s_high = max(0, self.s_high + diff - self.K)
        self.s_low = max(0, self.s_low - diff - self.K)
        
        alarm = (self.s_high > self.H) or (self.s_low > self.H)
        return alarm, self.s_high, self.s_low

In [60]:
def add_features(df, column_name, window=50):
    df['mean'] = df[column_name].rolling(window=window).mean()
    df['std'] = df[column_name].rolling(window=window).std()
    df['slope'] = df[column_name].diff()
    df = df.fillna(0)
    return df

In [56]:
def difference(df, column):
    data=df[column]
    results=[]
    results.append({"diff": 0})
    for i in range(1,len(data)):
        diff=data[i]-data[i-1]
        results.append({"diff": diff})
    return pd.DataFrame(results)

In [58]:
def accelerate(df, column):
    data=df[column]
    results=[]
    for i in range(2):
        results.append({"acc": 0})
    for i in range(2,len(data)):
        acc=(data[i]-data[i-1])-(data[i-1]-data[i-2])
        results.append({"acc": acc})
    return pd.DataFrame(results)

In [41]:

def analyze_sensor_data(df, sensor_column):
    data = df[sensor_column].values
    
    training_points = data[:200]
    
    deltas = np.diff(training_points) 
    
    target_slope = np.mean(deltas)  
    sigma_slope = np.std(deltas)
    
   
    if sigma_slope == 0:
        sigma_slope = 0.0001 


    detector = TrendingCusumDetector(target_slope=target_slope, sigma_slope=sigma_slope, h_factor=10)
    
    results = []
    for val in data:
        alarm, sh, sl = detector.update(val)
        results.append({
            'val': val,
            's_high': sh,
            's_low': sl,
            'alarm': alarm
        })
    
    return pd.DataFrame(results)

analyze_sensor_data(df, 'Температура масла в магистрали общей откачки (поз. Тм)')

,val,s_high,s_low,alarm
0,21.548,0.000000,0.0,False
1,21.548,0.000000,0.0,False
2,21.548,0.000000,0.0,False
3,21.548,0.000000,0.0,False
4,21.548,0.000000,0.0,False
...,...,...,...,...
12654,43.561,0.000000,0.0,False
12655,43.561,0.000000,0.0,False
12656,43.561,0.000000,0.0,False
12657,100.000,56.427283,0.0,True


In [57]:
difference(df, 'Температура масла в магистрали общей откачки (поз. Тм)')

,diff
0,0.000
1,0.000
2,0.000
3,0.000
4,0.000
...,...
12654,0.000
12655,0.000
12656,0.000
12657,56.439


In [59]:
accelerate(df, 'Температура масла в магистрали общей откачки (поз. Тм)')

,acc
0,0.000
1,0.000
2,0.000
3,0.000
4,0.000
...,...
12654,0.000
12655,0.000
12656,0.000
12657,56.439


In [61]:
add_features(df, 'Температура масла в магистрали общей откачки (поз. Тм)')

,Температура масла в магистрали общей откачки (поз. Тм),Температура слива масла из опоры турбины (поз. Т606),Температура масла на входе в двигатель за фильтром (поз. Т607),Температура топливного газа перед СК (перед расходомерным устройством) (поз. Ттг),Давление масла в нагнетающей магистрали двигателя (после фильтра) (поз. Pм),Давление масла в магистрали общей откачки (поз. Р615),Давление за ОК Pk1,Давление за ОК Pk2,Давление топливного газа перед дозаторами (поз. Pтг1.1),Вибрация промежуточного корпуса газогенератора (гориз.) (поз. В1),...,Температура воздуха на входе в газогенератор (Расчетная),Давление воздуха за компрессором (Расчетная),Частота вращения РНД приведенная,Частота вращения РВД приведенная,Температура газов за ТНД максимальная (Расчетная),Положение лопаток НА КВД. (поз. A2),Положение КПВ (поз. A3),mean,std,slope
0,21.548,28.160,24.361,8.762,302.095,198.239,244.713,257.590,2917.776,1.977,...,5.601,250.959,4119.418,9722.291,439.119,16.139,8.072,0.00000,0.000000,0.000
1,21.548,28.160,24.361,8.762,302.095,198.239,244.713,257.590,2917.776,1.977,...,5.601,251.152,4119.683,9722.339,439.119,16.139,8.072,0.00000,0.000000,0.000
2,21.548,28.160,24.361,8.762,302.559,198.220,243.865,258.663,2916.046,1.972,...,5.601,251.152,4119.905,9722.395,439.119,15.876,7.931,0.00000,0.000000,0.000
3,21.548,28.160,24.361,8.762,302.559,198.220,243.865,258.663,2916.046,1.972,...,5.601,251.264,4119.905,9722.395,439.107,15.876,7.931,0.00000,0.000000,0.000
4,21.548,28.230,24.361,8.762,302.262,197.553,243.865,257.820,2915.057,1.977,...,5.601,251.264,4119.773,9722.260,439.107,15.758,7.931,0.00000,0.000000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12654,43.561,65.943,23.177,23.477,308.473,199.926,935.612,941.988,2859.809,3.783,...,4.822,939.183,7834.907,10856.534,586.625,79.420,9.020,43.52254,0.049755,0.000
12655,43.561,65.943,23.107,23.477,308.473,199.926,933.762,942.065,2860.674,3.786,...,4.822,938.800,7837.355,10857.264,586.625,79.667,9.014,43.52300,0.050007,0.000
12656,43.561,65.943,23.107,23.477,308.473,199.926,933.762,942.065,2860.674,3.786,...,4.822,937.913,7837.810,10858.309,586.625,79.667,9.014,43.52346,0.050253,0.000
12657,100.000,65.943,23.107,23.477,308.232,200.667,934.841,942.065,2857.584,3.778,...,4.822,937.913,7838.946,10857.461,586.625,79.637,8.975,44.65270,7.987189,56.439
